# Stellar Mergers: A Binary Population Synthesis Approach

This notebook is a tutorial introducing you to binary population synthesis (BPS), binary stellar evolution, and how to run your own BPS simulations using `COMPAS`, an open source code for simulation populations of binary star systems.


In [27]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import subprocess
import pandas as pd

pd.set_option('display.max_columns', None)

## Binary Population Synthesis

### Massive Stars

Observational surveys have found that the majority of massive stars ($M>8 \ M_\odot$) are born in binaries or higher-order multiples, meaning that two or more stars are gravitationally bound and orbit a common center of mass (Sana et al. 2012). This is significant because now when studying the massive stars, we need to consider this binarity and the implications it has on the evolution of the stars because as a result of these companions, stars in binaries don't evolve in isolation--binary interactions impact evolution. Over the course of their lifetime, binary stars can:
- Transfer mass to their companion (mass transfer / Roche lobe overflow)
- Undergo runaway mass transfer that can lead to a stellar merger (common envelope)
- Become unbound after a supernova kick
- Remain bound and form a double compact object (binary black hole, binary neutron star, black hole-neutron star pair) which we can detect via gravitational waves when they merge

<img src="isolated_binary_evolution.png" width=600 style="display: block; margin: 0 auto;">

### BPS Codes

We can predict the statistical properties of binary star populations via binary population synthesis. Rather than studying individual massive binaries, we can simulate a population, analyze population trends, and compare them to observations from surveys. Binary population synthesis simulations do the following:
1. Draw initial parameters from distributions (masses, orbital period, mass ratios, metallicity) motivated by observations
2. Evolve each binary from the zero-age main sequence (ZAMS, aka birth) to its final state using rapid analytic stellar evolution prescriptions
3. Record outcomes: stellar merger, unbound by SNe, compact object binaries, etc
4. Collect statistics of the full population regarding outcomes + binary properties
5. (Post processing) Convolve these simulations with star formation histories to predict observable rates

We use these codes because detailed stellar evolution codes such as `MESA` can take days to evolve a single system. BPS codes can evolve a single binary in milliseconds, allowing us to explore large and multidimensional parameter spaces. However, the trade off is that BPS codes use simply and analytic fits. The strength is to use BPS results to do population inference.

### Stellar Mergers

A lot of BPS codes have been used to forward model the gravitational wave observations we have been collecting from the LIGO-Virgo-KAGRA (LVK) collaboration since 2016. This focus on the rare outcome of binary stellar evolution that results in binary black holes (BBHs), binary neutron stars (BNSs), and black hole-neutron star (BHNS) pairs, that are close enough to merge within the age of the universe. However, we want to use BPS codes to study stellar mergers: mergers of two massive stars during runaway mass transfer (common envelope phase). Specifically, we are interested in:
- Merger rates across cosmic time: predicting when and how often stellar mergers occur as a function of redshift, which depends on star formation rates and metallicitiy. 
- Which stars merge during common envelope: analyzing which binary configurations, e.g. masses, mass ratios, stellar types, merge during the common envelope phase

## `COMPAS`

`COMPAS` (Compact Object Mergers: Population Astrophysics and Statistics) is a publicly available rapid binary population synthesis code we will use to create and evolve our populations of binary stars (https://compas.science). Before continuing this tutorial that will get you familiar with using `COMPAS` you must first install `COMPAS` on your machine. The instructions for installing `COMPAS` can be found here: https://compas.readthedocs.io/en/latest/pages/Getting%20started/getting-started.html.

Once you have the executable installed you should be able to run this and get the following

```bash
# print COMPAS version
./COMPAS -v
```
```bash

COMPAS v03.27.03 (gsl v2.7.1, boost v1.83.0, HDF5 v1.10.10)
Compact Object Mergers: Population Astrophysics and Statistics
by Team COMPAS (http://compas.science/index.html)
A binary star simulator

Go to https://compas.readthedocs.io/en/latest/index.html for the online documentation
Check https://compas.readthedocs.io/en/latest/pages/whats-new.html to see what's new in the latest release
```

`COMPAS` works by:
1. Sampling initial conditions/population property distributions specified by the user
2. Evolves stars (mass, radius, stellar type) through each evolutionary phase using analytic formulas
3. Simulates binary interactions such as mass transfer, supernova, common envelopes, etc. via parameterized prescriptions which the user can specify (e.g. changing the knob off mass transfer efficiency)
4. Records outcomes of binaries after binary interactions

After evolving the population, `COMPAS` will spit out the results in HDF5 formats (file format to store LARGE amountso of data). `COMPAS` produces `COMPAS_Output.h5` which has the following data groups:
- `BSE_System_Parameters`: data group containing initial + final states of binaries
- `BSE_Common_Envelopes`: details of each binary's common envelope event(s)
- `BSE_RLOF`: details of each binary's mass transfer event(s) (everytime a companion overflows its Roche lobe)
- `BSE_Supernovae`: details of each binary's supernova event(s)
- `BSE_Double_Compact_Objects`: details for binary's ending as BBH, BNS, BHNS
<!-- - `BSE_Detailed_Output`: if enabled, you get this output for specific time steps -->

### Running a `COMPAS` Simulation

The easiest and most flexible way to run `COMPAS` with Python is to do so with a `YAML` config file that gets passed to the `compas_run_submit` command to run a `COMPAS` simulation with all your user-specified parameters. In this folder the `compas_tutorial_config.yaml` mirrors the `COMPAS` `compasConfigDefault.yaml` file. Usually this file is all commented out to run `COMPAS` with default values. In this tutorial I change the following:

```yaml
# ./tutorial/compasConfigDefault.yaml
booleanChoices:
    ...
    --detailed-output: False
    ...

numericalChoices:
    ...
    --number-of-systems: 10000
    --metallicity: 0.014200  
    --mass-transfer-fa: 0.50  
    --common-envelope-alpha: 1.00
    --random-seed: 42  
    ...
stringChoices:
    ...
    --mode: 'BSE'  
    --logfile-type: 'HDF5' 
    ...
```

Now, we can run `COMPAS` with this configuration file.

In [18]:
! compas_run_submit compas_tutorial_config.yaml

python_version = 3
compas_executable_override /home/cal/analam/Documents/COMPAS/src/COMPAS
grid_filename None
/home/cal/analam/Documents/COMPAS/src/COMPAS --detailed-output False --number-of-systems 1000 --metallicity 0.0142 --random-seed 42 --mass-transfer-fa 0.5 --common-envelope-alpha 1.0 --mode BSE --logfile-type HDF5 --output-path /home/cal/analam/Documents/stellar_mergers/tutorial



COMPAS v03.27.03 (gsl v2.7.1, boost v1.83.0, HDF5 v1.10.10)
Compact Object Mergers: Population Astrophysics and Statistics
by Team COMPAS (http://compas.science/index.html)
A binary star simulator

Go to https://compas.readthedocs.io/en/latest/index.html for the online documentation
Check https://compas.readthedocs.io/en/latest/pages/whats-new.html to see what's new in the latest release

Start generating binaries at Wed Apr  1 22:12:45 2026

0: Stars merged: (Main_Sequence_>_0.7 -> Early_Asymptotic_Giant_Branch) + (Main_Sequence_<=_0.7 -> Main_Sequence_<=_0.7)
1: Allowed time exceeded: (Main_Sequence_>_0.7 -> Carbon-Oxygen_White_Dwarf) + (Main_Sequence_>_0.7 -> Neutron_Star)
2: Double White Dwarf formed: (Main_Sequence_>_0.7 -> Oxygen-Neon_White_Dwarf) + (Main_Sequence_>_0.7 -> Carbon-Oxygen_White_Dwarf)
3: Allowed time exceeded: (Main_Sequence_>_0.7 -> Neutron_Star) + (Main_Sequence_<=_0.7 -> Main_Sequence_<=_0.7)
4: Double White Dwarf formed: (Main_Sequence_>_0.7 -> Oxygen-Neon_Whi

We evolved a population of $N=1,000$ binaries in $\sim 16$ seconds! Each numbered line tells you the final evolutionary state of the binary, the initial stellar type of the binary components, and the final stellar type of the binary components. I see some mergers!

### Key Parameters for this Project

The parameters we will be changing/exploring their impact are the following:
- `--number-of-systems`: number of binaries to evolve
- `--metallicity`: metallicity, default is solar (Z=0.0142)
- `--common-envelope-alpha`: common envelope energy efficiency, default is $\alpha_\text{CE}=1$
- `--mass-transfer-fa`: mass transfer accretion fraction, default is $\beta=0.5$
- `--random-seed`: seed number for simulation reproducability

### `COMPAS` Outputs

In the `COMPAS_outputs` folder you will find the simulation data. There is a file `RunDetails` with the simulation run details and the `COMPAS_Output.h5` which has our simulation data. Let's explore with this helper function.

In [28]:
def load_compas_group(h5_path, group_name):
    """
    Load a COMPAS HDF5 output group into a dictionary.
    
    Parameters:
    -----------
    h5_path : str
        Path to COMPAS_Output.h5
    group_name : str
        Name of the HDF5 group (e.g., 'BSE_System_Parameters')
    
    Returns:
    --------
    data : dict
        Dictionary mapping column names to numpy arrays
    """
    data = {}
    with h5py.File(h5_path, 'r') as f:
        group = f[group_name]
        for col in group.keys():
            data[col] = group[col][:]
    return data

output_path = 'COMPAS_Output/COMPAS_Output.h5'
sys_params_df = pd.DataFrame(load_compas_group(output_path, 'BSE_System_Parameters'))
sys_params_df.head(10)

,CH_on_MS(1),CH_on_MS(2),Eccentricity@ZAMS,Equilibrated_At_Birth,Error,Evolution_Status,Mass@ZAMS(1),Mass@ZAMS(2),Merger,Merger_At_Birth,Metallicity@ZAMS(1),Metallicity@ZAMS(2),Omega@ZAMS(1),Omega@ZAMS(2),PO_CE_Alpha,PO_CE_Formalism,PO_LBV_Factor,PO_Sigma_Kick_CCSN_BH,PO_Sigma_Kick_CCSN_NS,PO_Sigma_Kick_ECSN,PO_Sigma_Kick_USSN,PO_WR_Factor,Record_Type,SEED,SN_Kick_Magnitude_Random_Number(1),SN_Kick_Magnitude_Random_Number(2),SemiMajorAxis@ZAMS,Stellar_Type(1),Stellar_Type(2),Stellar_Type@ZAMS(1),Stellar_Type@ZAMS(2),Unbound
0,0,0,0.0,0,0,12,5.416431,0.365624,1,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,42,0.796543,0.950714,1.978083,5,0,1,0,0
1,0,0,0.0,0,43,3,9.884272,7.278885,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,43,0.496861,0.609067,0.179082,11,13,1,1,0
2,0,0,0.0,0,0,15,6.263390,2.504365,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,44,0.432542,0.104796,25.097195,12,11,1,1,0
3,0,0,0.0,0,0,3,8.209055,0.476427,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,45,0.304536,0.549545,39.662997,13,0,1,0,1
4,0,0,0.0,0,0,15,7.763171,3.106212,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,46,0.258047,0.634834,366.109909,12,11,1,1,0
5,0,0,0.0,0,43,14,35.560663,23.082648,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,47,0.851757,0.974483,0.950272,14,14,1,1,0
6,0,0,0.0,0,43,3,7.327586,6.346153,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,48,0.426484,0.891573,2.499422,13,12,1,1,0
7,0,0,0.0,0,0,15,5.141652,2.835602,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,49,0.473737,0.247062,49.968815,11,11,1,1,0
8,0,0,0.0,0,43,14,45.432704,18.814374,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,50,0.129175,0.228083,105.538949,14,14,1,1,0
9,0,0,0.0,0,0,3,8.613415,1.430632,0,0,0.0142,0.0142,0.0,0.0,1.0,0,1.5,217.0,217.0,30.0,30.0,1.0,1,51,0.327111,0.044712,135.193485,13,11,1,1,1


In [29]:
ce_df = pd.DataFrame(load_compas_group(output_path, 'BSE_Common_Envelopes'))
ce_df.head(10)

,BE_Fixed(1),BE_Fixed(2),BE_Kruckow(1),BE_Kruckow(2),BE_Loveridge(1),BE_Loveridge(2),BE_Loveridge_Winds(1),BE_Loveridge_Winds(2),BE_Nanjing(1),BE_Nanjing(2),Binding_Energy<CE(1),Binding_Energy<CE(2),CE_Event_Counter,Double_Core_CE,Eccentricity<CE,Eccentricity>CE,Immediate_RLOF>CE,Lambda@CE(1),Lambda@CE(2),Lambda_Convective(1),Lambda_Convective(2),Lambda_Fixed(1),Lambda_Fixed(2),Lambda_Kruckow(1),Lambda_Kruckow(2),Lambda_Loveridge(1),Lambda_Loveridge(2),Lambda_Loveridge_Winds(1),Lambda_Loveridge_Winds(2),Lambda_Nanjing(1),Lambda_Nanjing(2),Luminosity<CE(1),Luminosity<CE(2),MT_History,Mass(1)<CE,Mass(1)>CE,Mass(2)<CE,Mass(2)>CE,Mass_Convective_Env(1),Mass_Convective_Env(2),Mass_Env(1),Mass_Env(2),Merger,Optimistic_CE,RLOF(1),RLOF(2),Radius(1)<CE,Radius(1)>CE,Radius(2)<CE,Radius(2)>CE,Record_Type,RocheLobe(1)<CE,RocheLobe(1)>CE,RocheLobe(2)<CE,RocheLobe(2)>CE,SEED,SemiMajorAxis<CE,SemiMajorAxis>CE,SemiMajorAxisStage1>CE,Simultaneous_RLOF,Stellar_Type(1),Stellar_Type(1)<CE,Stellar_Type(2),Stellar_Type(2)<CE,Tau_Circ,Tau_Dynamical<CE(1),Tau_Dynamical<CE(2),Tau_Radial<CE(1),Tau_Radial<CE(2),Tau_Sync(1),Tau_Sync(2),Tau_Thermal<CE(1),Tau_Thermal<CE(2),Teff<CE(1),Teff<CE(2),Time,Zeta_Lobe,Zeta_Star
0,4.307984e+31,1.465026e+49,9.566597e+30,3.891172e+46,4.307984e+30,1.465026e+48,4.307984e+30,1.465026e+48,3.762642e+30,1.465026e+48,2.631717e+47,1.465026e+48,1,0,0.0,0.0,0,1.144936,1.000000,1.216789,1.211233,0.1,0.1,0.450315,37.649993,1.0,1.0,1.0,1.0,1.144936,1.000000,12596.076581,0.019605,6,5.311054,6.011000e+175,0.365624,4.000077e+01,2.220446e-16,0.337921,3.957640,0.000000,1,0,1,0,264.737275,1.759429e+97,0.346292,-8.465186e+01,1,262.892047,0.000000,80.265520,0.000000,42,433.247197,0.000000,0.0,0,5,5,0,0,0.0,9.345494e-08,1.685065e-11,0.276038,1.235713e+06,0.0,0.0,1.979230e-04,618.301646,3762.086814,3674.051785,98.720806,26.238545,0.063255
1,0.000000e+00,8.216967e+50,0.000000e+00,2.673226e+48,0.000000e+00,8.216967e+49,0.000000e+00,8.216967e+49,0.000000e+00,2.537386e+50,0.000000e+00,2.416518e+48,1,0,0.0,0.0,0,1.000000,0.831105,0.906687,0.914517,0.1,0.1,748.957830,30.738020,1.0,1.0,1.0,1.0,1.000000,0.323836,0.223016,49241.202572,4,1.315709,1.315709e+00,12.715627,3.188120e+00,0.000000e+00,0.000000,0.000000,9.527507,0,0,0,1,0.003903,3.903048e-03,228.923040,4.694357e-01,1,79.219107,0.972571,218.296044,1.455166,43,379.340589,3.183517,0.0,0,11,11,7,4,0.0,1.062909e-14,4.856634e-08,-1.000000,1.725185e-02,0.0,0.0,1.062909e-14,0.000337,63556.424229,5688.725109,37.121529,16.560896,0.056754
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.324666e+46,1,0,0.0,0.0,0,1.000000,11.072503,0.906455,0.906307,0.1,0.1,31405.664982,615.909772,1.0,1.0,1.0,1.0,1.000000,1.000000,0.000069,33009.978480,4,1.260000,1.260000e+00,6.416981,1.224256e+00,0.000000e+00,0.000000,0.000000,5.192725,0,0,0,1,0.000014,1.437401e-05,491.290184,5.233767e-03,1,234.816973,28.184174,489.689476,27.815993,48,937.506536,73.893840,0.0,0,13,13,12,6,0.0,2.427458e-18,2.149373e-07,-1.000000,8.440440e-01,0.0,0.0,2.427458e-18,0.000065,138781.915354,3513.741471,69.480774,7.514170,-0.035424
3,2.200934e+50,2.547846e+50,4.406934e+47,3.259769e+48,2.200934e+49,2.547846e+49,2.200934e+49,2.547846e+49,8.792098e+49,2.547846e+49,8.703113e+47,2.547846e+49,1,0,0.0,0.0,0,0.931502,1.000000,0.905983,0.921980,0.1,0.1,49.942520,7.816031,1.0,1.0,1.0,1.0,0.250331,1.000000,5503.861829,803.198623,3,6.153618,1.146533e+00,4.957728,4.957728e+00,0.000000e+00,0.000000,5.007085,0.000000,0,0,1,0,144.236662,2.266649e-01,3.661092,3.661092e+00,1,142.769087,2.739278,129.348904,5.313432,54,358.931799,10.455587,0.0,0,7,3,1,1,0.0,3.491550e-08,1.573057e-10,0.075200,1.194364e+02,0.0,0.0,1.218715e-03,0.262459,4143.874435,16075.998972,64.727500,0.187278,-0.042149
4,0.000000e+00,-3.780908e+32,0.000000e+00,-9.350353e+30,0.000000e+00,-3.780908e+31,0.000000e+00,-3.780908e+31,0.000000e+00,-4.081377e+31,0.000000e+00,9.156910e+48,2,0,0.0,0.0,0,1.00000